### Simultane Labyrinthe – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024/2025), 2. Runde, Aufgabe 2*

Anton, Bea und Chris haben von der neuesten Attraktion im Nachbardorf gehört: Bauer Knuth hat in seinem Maisfeld zwei Labyrinthe
angelegt. Beide haben dieselbe Grundgröße, ein rechteckiges Gitter mit vorgegebenen Seitenlängen n und m. Zwei Personen sollen in
beiden Labyrinthen gleichzeitig von Feld (1,1) starten und sich zum Zielfeld (n, m) bewegen. (In den Beispieldateien beginnen die
Koordinaten bei (0,0).) Dabei werden sie von einer dritten Person mit ein und derselben Anweisungsfolge geleitet. Eine Anweisung
gibt eine Richtung vor, in die sich beide Personen in einem Schritt um ein Gitterfeld bewegen sollen. Würde eine Person dabei gegen
eine Wand laufen, so ignoriert sie die entsprechende Anweisung. Erreicht eine Person das Zielfeld vor der anderen, so bleibt sie
dort stehen und wartet.

Chris leitet Anton und Bea mit einer Folge aus neun Anweisungen durch die beiden Labyrinthe der Aufgabenstellung (3 × 3 Felder, das ist
`labyrinthe0.txt`):

1) ↓ 2) → 3) ↓ 4) → 5) ↑ 6) ↑ 7) → 8) ↓ 9) ↓

(Auf dem Aufgabenblatt zeigen Bilder, wie sich beide dabei bewegen.) Chris hätte stattdessen Bea mit sechs Anweisungen direkt zum
Zielfeld leiten können. Dann wäre Anton jedoch im linken Labyrinth noch fast am Start und Chris bräuchte weitere sechs Anweisungen,
damit auch Anton das Zielfeld erreicht. Insgesamt wären dann 12 Anweisungen erforderlich.

Bauer Knuths Art, Gäste anzulocken, macht die Runde, und bald sprießen die Labyrinthpaare aus den Ackerböden.

#### Aufgabe

a) Entwirf einen Algorithmus, der für ein gegebenes Labyrinthpaar eine möglichst kurze Anweisungsfolge berechnet, mit der beide
Personen das Zielfeld erreichen.

b) Bald gibt es auch Labyrinthe mit Gruben: Würde eine Person durch eine Anweisung in eine Grube fallen, so geht sie stattdessen zum
Startfeld zurück. Erweitere deinen Algorithmus so, dass er auch für solche Labyrinthe funktioniert.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `labyrinthe0.txt` bis `labyrinthe9.txt`:

- 1. Zeile: Breite `n` und Höhe `m` (beide Labyrinthe sind gleich groß).
- Dann kommt zuerst das 1., danach das 2. Labyrinth. Jedes besteht aus `m` Zeilen mit je `n − 1` Zahlen (für jedes Feld: 1, wenn
  rechts davon eine senkrechte Wand ist, sonst 0), dann `m − 1` Zeilen mit je `n` Zahlen (1, wenn unterhalb des Feldes eine
  waagerechte Wand ist), dann einer Zeile mit der Anzahl `g` der Gruben und `g` Zeilen `x y` mit den Koordinaten der Gruben.
- Das Startfeld ist `(0,0)` oben links, das Zielfeld `(n−1, m−1)` unten rechts.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**

- **Zustandsraum und Breitensuche.** Ein *Zustand* beschreibt alles, was man zum Weitermachen wissen muss. Hier sind das die Felder
  *beider* Personen, also ein Paar `(a, b)`. Jede Anweisung führt von einem Zustand zu genau einem neuen Zustand. Die Breitensuche
  probiert zuerst alle Zustände, die man mit 0 Anweisungen erreicht, dann alle mit 1 Anweisung, dann mit 2 usw. Der erste Weg zum
  Zielzustand ist dadurch automatisch ein kürzester. Ein Zustand wird nur einmal besucht. Damit man die Anweisungen zurückverfolgen
  kann, merkt man sich zu jedem Zustand seinen **Vorgänger** und die Anweisung, mit der man von dort kam.
- **Abstände vom Ziel aus.** Eine Breitensuche *rückwärts* vom Zielfeld liefert für jedes Feld eines einzelnen Labyrinths, wie viele
  Anweisungen eine Person allein bis zum Ziel braucht.
- **A\*-Suche.** Eine Verbesserung der Breitensuche: Man bevorzugt Zustände, die *vielversprechend* aussehen. Zu jedem Zustand rechnet
  man `f = g + h`, dabei ist `g` die Zahl der bisherigen Anweisungen und `h` eine **untere Schranke** für die noch fehlenden. Hier ist
  `h` der größere der beiden Einzelabstände zum Ziel: Wenn Bea allein noch 5 Anweisungen braucht und Anton noch 3, dann braucht man
  zusammen mindestens 5. Man nimmt immer den Zustand mit dem kleinsten `f`.

**Python-Werkzeuge**

- `collections.deque` als Warteschlange (`append`, `popleft`).
- `heapq.heappush` und `heapq.heappop` als Prioritätswarteschlange (liefert immer das kleinste Element, Tupel werden von links nach
  rechts verglichen).
- Dictionaries (`dict`) mit Tupeln als Schlüssel für „besucht“ und „Vorgänger“, `set` für „fertig bearbeitet“.
- Für Neugierige: `numpy` kann eine ganze Schicht der Breitensuche auf einmal bearbeiten.

## Die Idee

1. **Einlesen** und für jedes Labyrinth eine **Zugtabelle** bauen: `tab[r][feld]` ist das Feld, auf dem man nach der Anweisung `r`
   steht, wenn man auf `feld` war. Wände, Gruben und das Ziel stecken dann nur in dieser Tabelle, die Suche selbst muss davon nichts
   mehr wissen.
2. Eine Anweisungsfolge **nachspielen** können, um jede gefundene Lösung unabhängig zu prüfen.
3. **Breitensuche** über die Zustände `(a, b)`. Das ist die vollständige Lösung zu a) und b), denn die Gruben sind schon in der
   Zugtabelle. Sie ist für kleine Beispiele schnell genug.
4. **A\*** mit den Einzelabständen als Schranke spart je nach Beispiel viele Zustände.
5. Für die ganz großen Beispiele reicht reines Python nicht. Das zeigen wir ehrlich und rechnen sie dann für Neugierige mit `numpy`.

## Schritt 1: Einlesen

Die Datei wird in Zahlenlisten zerlegt. Für jedes Labyrinth lesen wir die senkrechten Wände (`m` Zeilen), die waagerechten Wände
(`m − 1` Zeilen) und die Gruben. `senkrecht[y][x] == 1` heißt: Rechts vom Feld `(x, y)` ist eine Wand. `waagerecht[y][x] == 1`
heißt: Unter dem Feld `(x, y)` ist eine Wand.

In [ ]:
def lies(name):
    with open('beispieldaten/' + name + '.txt', encoding='utf-8') as f:
        zeilen = [z.split() for z in f.read().split('\n')]
    n, m = int(zeilen[0][0]), int(zeilen[0][1])
    zeile = 1
    labyrinthe = []
    for _ in range(2):
        senkrecht = []
        for y in range(m):
            senkrecht.append([int(x) for x in zeilen[zeile]])
            zeile += 1
        waagerecht = []
        for y in range(m - 1):
            waagerecht.append([int(x) for x in zeilen[zeile]])
            zeile += 1
        anzahl_gruben = int(zeilen[zeile][0])
        zeile += 1
        gruben = []
        for _ in range(anzahl_gruben):
            gruben.append((int(zeilen[zeile][0]), int(zeilen[zeile][1])))
            zeile += 1
        labyrinthe.append((senkrecht, waagerecht, gruben))
    return n, m, labyrinthe


n, m, labyrinthe = lies('labyrinthe0')
print('Größe:', n, 'x', m)
for senkrecht, waagerecht, gruben in labyrinthe:
    print('senkrecht:', senkrecht, ' waagerecht:', waagerecht, ' Gruben:', gruben)

Größe: 3 x 3
senkrecht: [[1, 0], [1, 1], [0, 1]]  waagerecht: [[0, 0, 0], [0, 0, 0]]  Gruben: []
senkrecht: [[1, 0], [0, 1], [0, 0]]  waagerecht: [[0, 0, 0], [1, 1, 0]]  Gruben: [(0, 2)]


## Schritt 2: Die Zugtabelle

Wir nummerieren die Felder zeilenweise: `feld = y * n + x`. Start ist Feld 0, Ziel ist Feld `n * m − 1`. Die vier Anweisungen sind
`0 = ↑`, `1 = →`, `2 = ↓`, `3 = ←`.

Für jede Anweisung `r` und jedes Feld legen wir fest, wo die Person danach steht:

- Auf dem **Ziel** bleibt sie stehen.
- Läuft sie aus dem Gitter hinaus oder gegen eine **Wand**, bleibt sie stehen. Eine Wand zwischen `(x, y)` und dem rechten Nachbarn
  steht in `senkrecht[y][x]`, die zwischen `(x, y)` und dem linken Nachbarn in `senkrecht[y][x − 1]`. Entsprechend bei `waagerecht`
  nach unten (`[y][x]`) und nach oben (`[y − 1][x]`).
- Würde sie in eine **Grube** laufen, steht sie danach auf dem Startfeld 0.
- Auf einer Grube kann nie jemand stehen. Für diese Felder tragen wir `None` ein.

In [ ]:
PFEILE = '↑→↓←'
RICHTUNGEN = [(0, -1), (1, 0), (0, 1), (-1, 0)]      # (dx, dy) zu ↑ → ↓ ←


def zugtabelle(n, m, labyrinth):
    senkrecht, waagerecht, gruben = labyrinth
    ziel = n * m - 1
    grubenfelder = set()
    for x, y in gruben:
        grubenfelder.add(y * n + x)
    tabelle = []
    for r in range(4):
        dx, dy = RICHTUNGEN[r]
        zeile = []
        for feld in range(n * m):
            x, y = feld % n, feld // n
            if feld in grubenfelder:
                neu = None
            elif feld == ziel:
                neu = feld
            else:
                nx, ny = x + dx, y + dy
                if nx < 0 or nx >= n or ny < 0 or ny >= m:
                    neu = feld
                elif r == 1 and senkrecht[y][x] == 1:
                    neu = feld
                elif r == 3 and senkrecht[y][x - 1] == 1:
                    neu = feld
                elif r == 2 and waagerecht[y][x] == 1:
                    neu = feld
                elif r == 0 and waagerecht[y - 1][x] == 1:
                    neu = feld
                else:
                    neu = ny * n + nx
                    if neu in grubenfelder:
                        neu = 0
            zeile.append(neu)
        tabelle.append(zeile)
    return tabelle


def lade(name):
    n, m, labyrinthe = lies(name)
    tab1 = zugtabelle(n, m, labyrinthe[0])
    tab2 = zugtabelle(n, m, labyrinthe[1])
    return n, m, tab1, tab2


n, m, tab1, tab2 = lade('labyrinthe0')
print('1. Labyrinth, nach → von Feld 0 bis 8:', tab1[1])
print('2. Labyrinth, nach ↓ von Feld 0 bis 8:', tab2[2], ' (Feld 6 ist die Grube)')

1. Labyrinth, nach → von Feld 0 bis 8: [0, 2, 2, 3, 4, 5, 7, 7, 8]
2. Labyrinth, nach ↓ von Feld 0 bis 8: [3, 4, 5, 3, 4, 8, None, 7, 8]  (Feld 6 ist die Grube)


## Schritt 3: Anweisungen nachspielen

`fuehre_aus` setzt beide Personen auf das Startfeld und wendet die Anweisungen der Reihe nach auf beide Zugtabellen an. Das ist
unsere Kontrolle für jede Lösung. Mit `aus_pfeilen` und `als_pfeile` wandeln wir zwischen Pfeil-Text und Zahlenlisten um.

Die Folge der Aufgabenstellung (9 Anweisungen) muss beide Personen auf das Zielfeld 8 bringen:

In [ ]:
def fuehre_aus(folge, tab1, tab2):
    a = 0
    b = 0
    for r in folge:
        a = tab1[r][a]
        b = tab2[r][b]
    return a, b


def aus_pfeilen(text):
    folge = []
    for zeichen in text:
        if zeichen in PFEILE:
            folge.append(PFEILE.index(zeichen))
    return folge


def als_pfeile(folge):
    return ' '.join(PFEILE[r] for r in folge)


n, m, tab1, tab2 = lade('labyrinthe0')
chris = aus_pfeilen('↓ → ↓ → ↑ ↑ → ↓ ↓')
print(len(chris), 'Anweisungen, Endfelder:', fuehre_aus(chris, tab1, tab2), ' Ziel:', n * m - 1)

9 Anweisungen, Endfelder: (8, 8)  Ziel: 8


## Schritt 4: Breitensuche über die Zustände

Ein Zustand ist das Paar `(a, b)` der Felder. Start ist `(0, 0)`, Ziel ist `(ziel, ziel)`. Von jedem Zustand führen vier
Anweisungen zu neuen Zuständen. Das Dictionary `abstand` enthält alle schon gesehenen Zustände mit ihrer Anweisungszahl, in
`vorgaenger` steht zu jedem Zustand `(Zustand davor, Anweisung)`.

Die Rückgabe ist die kürzeste Folge (oder `None`, wenn es keine gibt) und die Zahl der besuchten Zustände. Diese Zahl nehmen wir
später statt einer Laufzeit, um die Verfahren zu vergleichen.

In [ ]:
from collections import deque


def rekonstruiere(vorgaenger, zustand):
    folge = []
    while vorgaenger[zustand] is not None:
        zustand, r = vorgaenger[zustand]
        folge.append(r)
    folge.reverse()
    return folge


def breitensuche(tab1, tab2):
    ziel = len(tab1[0]) - 1
    start = (0, 0)
    abstand = {start: 0}
    vorgaenger = {start: None}
    schlange = deque([start])
    while schlange:
        zustand = schlange.popleft()
        if zustand == (ziel, ziel):
            return rekonstruiere(vorgaenger, zustand), len(abstand)
        a, b = zustand
        for r in range(4):
            neu = (tab1[r][a], tab2[r][b])
            if neu not in abstand:
                abstand[neu] = abstand[zustand] + 1
                vorgaenger[neu] = (zustand, r)
                schlange.append(neu)
    return None, len(abstand)


n, m, tab1, tab2 = lade('labyrinthe0')
folge, besucht = breitensuche(tab1, tab2)
print(len(folge), 'Anweisungen:', als_pfeile(folge))
print('Besuchte Zustände:', besucht, ' von höchstens', (n * m) ** 2)
print('Nachgespielt:', fuehre_aus(folge, tab1, tab2))

8 Anweisungen: ↓ ↓ → ↑ ↑ → ↓ ↓
Besuchte Zustände: 32  von höchstens 81
Nachgespielt: (8, 8)


Das ist schon die Lösung für `labyrinthe0`: 8 statt der 9 Anweisungen von Chris. Zum Vergleich der Idee aus der Aufgabe: Einzeln
bräuchte man 6 + 6 = 12.

**Gibt es noch eine kürzere Folge?** Bei 8 Anweisungen gibt es nur 4⁸ = 65536 Folgen. Die probieren wir zur Kontrolle alle durch
(`itertools.product`, kartesisches Produkt). Wie oft kommen beide ans Ziel, und gibt es eine kürzere?

In [ ]:
from itertools import product

for laenge in range(1, 9):
    treffer = []
    for folge_test in product(range(4), repeat=laenge):
        if fuehre_aus(folge_test, tab1, tab2) == (8, 8):
            treffer.append(folge_test)
    print('Länge', laenge, ':', len(treffer), 'Folge(n)')
    if treffer:
        print(als_pfeile(treffer[0]))
        break

Länge 1 : 0 Folge(n)
Länge 2 : 0 Folge(n)
Länge 3 : 0 Folge(n)
Länge 4 : 0 Folge(n)
Länge 5 : 0 Folge(n)
Länge 6 : 0 Folge(n)
Länge 7 : 0 Folge(n)
Länge 8 : 1 Folge(n)
↓ ↓ → ↑ ↑ → ↓ ↓


Es gibt bei dieser Länge genau **eine** Folge, die wir auch gefunden haben. Die offiziellen Lösungshinweise nennen für `labyrinthe0` zwar
„2 optimale Lösungen“, dieselbe Folge `↓ ↓ → ↑ ↑ → ↓ ↓` steht aber auch dort. Wie die Hinweise zählen, steht nicht dabei. Mit diesem
vollständigen Durchprobieren sind wir uns bei diesem Beispiel sicher. Die Anzahl der Lösungen wird in der Aufgabe auch nicht
verlangt, wir gehen darauf nicht weiter ein.

## Schritt 5: Die kleinen Beispiele und die Gruben

Die Hinweise nennen für die Beispiele 0 bis 3 (und 7) die Anzahl der Anweisungen. `labyrinthe2` und `labyrinthe7` haben Gruben,
und `labyrinthe7` soll **keine Lösung** haben. Wir rechnen alle fünf mit der Breitensuche und spielen jede Lösung nach.

In [ ]:
HINWEISE_ANZAHL = {0: 8, 1: 31, 2: 65, 3: 164, 4: 14384, 5: 1308, 6: 1844, 7: None, 8: 472, 9: 1012}

ergebnis_bfs = {}
for nr in [0, 1, 2, 3, 7]:
    n, m, tab1, tab2 = lade('labyrinthe%d' % nr)
    folge, besucht = breitensuche(tab1, tab2)
    ergebnis_bfs[nr] = (folge, besucht)
    if folge is None:
        print('labyrinthe%d (%d x %d): keine Lösung, %d Zustände besucht' % (nr, n, m, besucht))
    else:
        ende = fuehre_aus(folge, tab1, tab2)
        print('labyrinthe%d (%d x %d): %d Anweisungen (Hinweise: %d), nachgespielt Endfelder %s, besuchte Zustände: %d' % (
            nr, n, m, len(folge), HINWEISE_ANZAHL[nr], ende, besucht))

labyrinthe0 (3 x 3): 8 Anweisungen (Hinweise: 8), nachgespielt Endfelder (8, 8), besuchte Zustände: 32
labyrinthe1 (5 x 4): 31 Anweisungen (Hinweise: 31), nachgespielt Endfelder (19, 19), besuchte Zustände: 342
labyrinthe2 (10 x 10): 65 Anweisungen (Hinweise: 65), nachgespielt Endfelder (99, 99), besuchte Zustände: 8206
labyrinthe3 (10 x 50): 164 Anweisungen (Hinweise: 164), nachgespielt Endfelder (499, 499), besuchte Zustände: 235706
labyrinthe7 (30 x 10): keine Lösung, 49056 Zustände besucht


Die Längen 8, 31, 65 und 164 stimmen mit den Lösungshinweisen überein. Zusätzlich haben wir die in den Hinweisen abgedruckten
Anweisungsfolgen dieser Beispiele mit `fuehre_aus` nachgespielt, und sie führen mit **unseren** Zugtabellen beide ans Ziel (das prüft
der Generator). Unsere Regeln für Wände, Gruben und das Ziel sind damit bestätigt.

Auffällig bei `labyrinthe2`: Hier braucht man die Grube. Ohne Gruben (wir lassen sie in beiden Labyrinthen weg) bräuchte man mehr
Anweisungen:

In [ ]:
n, m, labyrinthe = lies('labyrinthe2')
ohne_gruben = [(senkrecht, waagerecht, []) for senkrecht, waagerecht, gruben in labyrinthe]
folge_ohne, _ = breitensuche(zugtabelle(n, m, ohne_gruben[0]), zugtabelle(n, m, ohne_gruben[1]))
print('labyrinthe2 ohne Gruben:', len(folge_ohne), 'Anweisungen, mit Gruben:', len(ergebnis_bfs[2][0]))

labyrinthe2 ohne Gruben: 67 Anweisungen, mit Gruben: 65


Eine Grube wirkt hier wie eine **Abkürzung zurück zum Start**: Die Person muss sich nicht Feld für Feld zurückarbeiten.

**Und `labyrinthe7`?** Die Breitensuche besucht alle erreichbaren Zustände und findet das Ziel nicht. Das geht auch schneller: Wenn
*eine* Person allein nie ins Ziel kommt, kann es auch zu zweit nicht klappen. Umgekehrt reicht es, wenn jede Person einzeln ans Ziel
kommt. Dann führt man erst die eine ans Ziel (dort wartet sie) und danach die andere: Jede Position, auf der eine Person gerade steht,
erreicht man vom Start aus ohne Grube, und alle Schritte lassen sich rückwärts wieder zurückgehen. Von jedem Feld kommt man also zum
Start und von dort zum Ziel. Dafür berechnen wir als Nächstes die Abstände.

## Schritt 6: Abstände zum Ziel und A\*

**Abstände.** `distanzen` macht eine Breitensuche *rückwärts* vom Ziel. Dazu legen wir zu jedem Feld eine Liste der Felder an, von
denen man mit *einer* Anweisung auf dieses Feld kommt (`davor`). Für Felder, von denen aus das Ziel gar nicht erreichbar ist, bleibt
der Abstand `-1`, ebenso für Gruben.

Probe an `labyrinthe0`: Die Hinweise zeigen für die beiden Labyrinthe diese Abstandstabellen (die Grube hat dort keinen sinnvollen Wert):

```
8 3 2        6 3 2
7 4 1        5 4 1
6 5 0        x 1 0
```

In [ ]:
def distanzen(tab):
    felder = len(tab[0])
    ziel = felder - 1
    davor = [[] for _ in range(felder)]
    for r in range(4):
        for feld in range(felder):
            neu = tab[r][feld]
            if neu is not None and neu != feld:
                davor[neu].append(feld)
    dist = [-1] * felder
    dist[ziel] = 0
    schlange = deque([ziel])
    while schlange:
        feld = schlange.popleft()
        for vorher in davor[feld]:
            if dist[vorher] == -1:
                dist[vorher] = dist[feld] + 1
                schlange.append(vorher)
    return dist


def zeige_abstaende(dist, n):
    for y in range(len(dist) // n):
        print(' '.join('%3d' % d for d in dist[y * n:(y + 1) * n]))


n, m, tab1, tab2 = lade('labyrinthe0')
zeige_abstaende(distanzen(tab1), n)
print()
zeige_abstaende(distanzen(tab2), n)

  8   3   2
  7   4   1
  6   5   0

  6   3   2
  5   4   1
 -1   1   0


Die Zahlen stimmen mit den Hinweisen überein (nur auf der Grube steht bei uns `-1`). Die Lösbarkeit prüft man jetzt in einem Blick:
Ist der Abstand des Startfelds `-1`, gibt es keine Lösung. Bei `labyrinthe7` ist das im 2. Labyrinth der Fall. Welche Grube ist schuld? Wir lassen
nacheinander jede Grube weg und sehen nach, ob das Startfeld dann einen Abstand hat:

In [ ]:
n, m, labyrinthe = lies('labyrinthe7')
tab1, tab2 = zugtabelle(n, m, labyrinthe[0]), zugtabelle(n, m, labyrinthe[1])
print('Abstand des Startfelds im 1. Labyrinth:', distanzen(tab1)[0], ' im 2. Labyrinth:', distanzen(tab2)[0])
senkrecht, waagerecht, gruben = labyrinthe[1]
for grube in gruben:
    rest = [g for g in gruben if g != grube]
    dist = distanzen(zugtabelle(n, m, (senkrecht, waagerecht, rest)))
    print('ohne die Grube', grube, ': Abstand des Startfelds im 2. Labyrinth', dist[0])

Abstand des Startfelds im 1. Labyrinth: 74  im 2. Labyrinth: -1
ohne die Grube (15, 1) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (25, 1) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (2, 2) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (10, 5) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (14, 8) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (26, 9) : Abstand des Startfelds im 2. Labyrinth -1
ohne die Grube (18, 6) : Abstand des Startfelds im 2. Labyrinth 84


Nur wenn man die Grube bei `(18, 6)` wegnimmt, wird das 2. Labyrinth lösbar. Sie sperrt also den einzigen Weg zum Ziel. (In den Hinweisen
steht dafür die Koordinate `(14, 8)`. Nach unserer Rechnung ist es `(18, 6)`.)

**A\*-Suche.** Wir ersetzen die Warteschlange durch eine Prioritätswarteschlange (`heapq`). Ein Eintrag ist `(f, g, zustand)` mit
`f = g + h` und `h = max(dist1[a], dist2[b])`. Bei gleichem `f` entscheidet `g`, danach der Zustand selbst, weil Python Tupel von links nach
rechts vergleicht. Für die Richtigkeit ist diese Reihenfolge egal.

Warum ist das Ergebnis trotzdem kürzest? `h` ist eine echte untere Schranke, und eine Anweisung senkt den Einzelabstand einer Person
um höchstens 1. Damit ist der erste Zeitpunkt, an dem das Ziel aus der Warteschlange kommt, optimal. Ein Zustand muss also nur dann
bearbeitet werden, wenn er nicht schon bearbeitet wurde (`fertig`).

Statt der besuchten Zustände zählen wir hier die **bearbeiteten** Zustände (`len(fertig)`), das sind die, die aus der Warteschlange
genommen und aufgemacht wurden.

In [ ]:
import heapq


def astern(tab1, tab2):
    ziel = len(tab1[0]) - 1
    dist1 = distanzen(tab1)
    dist2 = distanzen(tab2)
    if dist1[0] == -1 or dist2[0] == -1:
        return None, 0                      # unlösbar, das sieht man schon an den Abständen
    start = (0, 0)
    beste = {start: 0}
    vorgaenger = {start: None}
    fertig = set()
    warteschlange = [(max(dist1[0], dist2[0]), 0, start)]
    while warteschlange:
        f, g, zustand = heapq.heappop(warteschlange)
        if zustand in fertig:
            continue
        fertig.add(zustand)
        if zustand == (ziel, ziel):
            return rekonstruiere(vorgaenger, zustand), len(fertig)
        a, b = zustand
        for r in range(4):
            neu = (tab1[r][a], tab2[r][b])
            if neu not in beste or g + 1 < beste[neu]:
                beste[neu] = g + 1
                vorgaenger[neu] = (zustand, r)
                h = max(dist1[neu[0]], dist2[neu[1]])
                heapq.heappush(warteschlange, (g + 1 + h, g + 1, neu))
    return None, len(fertig)


ergebnis_astern = {}
for nr in [0, 1, 2, 3, 7]:
    n, m, tab1, tab2 = lade('labyrinthe%d' % nr)
    folge, bearbeitet = astern(tab1, tab2)
    ergebnis_astern[nr] = (folge, bearbeitet)
    laenge = 'keine Lösung' if folge is None else '%d Anweisungen' % len(folge)
    print('labyrinthe%d: %s, A* bearbeitet %d Zustände, Breitensuche besuchte %d' % (nr, laenge, bearbeitet, ergebnis_bfs[nr][1]))

labyrinthe0: 8 Anweisungen, A* bearbeitet 9 Zustände, Breitensuche besuchte 32
labyrinthe1: 31 Anweisungen, A* bearbeitet 150 Zustände, Breitensuche besuchte 342
labyrinthe2: 65 Anweisungen, A* bearbeitet 3012 Zustände, Breitensuche besuchte 8206
labyrinthe3: 164 Anweisungen, A* bearbeitet 100283 Zustände, Breitensuche besuchte 235706
labyrinthe7: keine Lösung, A* bearbeitet 0 Zustände, Breitensuche besuchte 49056


Beide Verfahren liefern gleich lange Folgen. A\* bearbeitet weniger als die Hälfte der Zustände, bei `labyrinthe3` rund 100000 statt 236000
(ein genauer Vergleich ist schwierig, weil „besucht“ bei der Breitensuche auch noch nicht bearbeitete Zustände in der Warteschlange mitzählt). Bei den
kleinen Beispielen spielt das keine Rolle. Bei `labyrinthe6` (250 × 250 Felder, also 62500² ≈ 3,9 Milliarden Zustände) ist es der
Unterschied zwischen „geht“ und „geht nicht“:

In [ ]:
n, m, tab1, tab2 = lade('labyrinthe6')
folge6, bearbeitet6 = astern(tab1, tab2)
print('labyrinthe6 (%d x %d): %d Anweisungen (Hinweise: %d), bearbeitete Zustände: %d' % (
    n, m, len(folge6), HINWEISE_ANZAHL[6], bearbeitet6))
print('Nachgespielt:', fuehre_aus(folge6, tab1, tab2), ' Ziel:', n * m - 1)
print('Möglich wären', (n * m) ** 2, 'Zustände, bearbeitet wurden', round(100 * bearbeitet6 / (n * m) ** 2, 3), 'Prozent.')
print('Die ersten 30 Anweisungen:', als_pfeile(folge6[:30]))

labyrinthe6 (250 x 250): 1844 Anweisungen (Hinweise: 1844), bearbeitete Zustände: 485460
Nachgespielt: (62499, 62499)  Ziel: 62499
Möglich wären 3906250000 Zustände, bearbeitet wurden 0.012 Prozent.
Die ersten 30 Anweisungen: → → → ↓ ↓ ↓ ↓ ↓ ↓ ↓ ↓ → → ↓ → ↓ ↓ → → → ↓ → ↓ → → ↑ → → ↓ ↓


## Schritt 7: Alle Beispiele und die Grenze von reinem Python

Die Tabelle fasst zusammen. `?` heißt: Mit unserem Python-Code ohne Hilfsmittel nicht in vernünftiger Zeit gelöst. Die Zahl der
möglichen Zustände ist `(n·m)²`.

In [ ]:
print('Nr  Größe     mögliche Zustände   Hinweise   Breitensuche   A*      Zustände BFS / A*')
for nr in range(10):
    n, m, labyrinthe = lies('labyrinthe%d' % nr)
    moegliche = (n * m) ** 2
    soll = HINWEISE_ANZAHL[nr]
    if nr in ergebnis_bfs:
        bfs = ergebnis_bfs[nr][0]
        bfs_text = '-' if bfs is None else str(len(bfs))
        zustaende = '%d / %d' % (ergebnis_bfs[nr][1], ergebnis_astern[nr][1])
    else:
        bfs_text = '?'
        zustaende = ''
    if nr in ergebnis_astern:
        astern_folge = ergebnis_astern[nr][0]
        astern_text = '-' if astern_folge is None else str(len(astern_folge))
    elif nr == 6:
        astern_text = str(len(folge6))
        zustaende = '? / %d' % bearbeitet6
    else:
        astern_text = '?'
    print('%2d  %3d x %-3d %16d   %8s   %12s   %-5s   %s' % (
        nr, n, m, moegliche, '-' if soll is None else soll, bfs_text, astern_text, zustaende))

Nr  Größe     mögliche Zustände   Hinweise   Breitensuche   A*      Zustände BFS / A*
 0    3 x 3                 81          8              8   8       32 / 9
 1    5 x 4                400         31             31   31      342 / 150
 2   10 x 10             10000         65             65   65      8206 / 3012
 3   10 x 50            250000        164            164   164     235706 / 100283
 4  101 x 101        104060401      14384              ?   ?       
 5  151 x 151        519885601       1308              ?   ?       
 6  250 x 250       3906250000       1844              ?   1844    ? / 485460
 7   30 x 10             90000          -              -   -       49056 / 0
 8  150 x 150        506250000        472              ?   ?       
 9  170 x 170        835210000       1012              ?   ?       


**Was wir erreicht haben und was nicht:**

- Beispiele 0, 1, 2, 3 und 7: Breitensuche und A\* geben dieselben Längen wie die Hinweise (8, 31, 65, 164, keine Lösung).
- Beispiel 6: A\* findet 1844 Anweisungen wie die Hinweise, bei einigen Hunderttausend bearbeiteten Zuständen. Die Breitensuche
  haben wir dort nicht versucht (`?`). Die Hinweise nennen für sie 3 s statt 188 ms bei A\*.
- Beispiele 4, 5, 8 und 9 (101 × 101 bis 170 × 170 Felder) bleiben mit reinem Python **offen (`?`)**. Die Hinweise berichten
  A\*-Laufzeiten von 5 bis 19 Sekunden in einer schnelleren Sprache. Der Grund: Hier ist der kürzeste Weg sehr lang
  (14384 bzw. 1308, 472, 1012 Anweisungen), und fast alle der rund 100 bis 840 Millionen möglichen Zustände liegen vor dem Ziel. Eine
  Suche, die jeden Zustand als Python-Tupel in einem Dictionary und einer Warteschlange speichert, braucht dafür sehr viel Speicher und
  Zeit. Unsere A\*-Versuche für die Beispiele 4 und 8 waren nach vielen Minuten noch nicht fertig.

Darum der folgende Abschnitt für Neugierige.

## Für Neugierige: Breitensuche schichtweise mit numpy

Die Breitensuche bearbeitet alle Zustände mit gleicher Anweisungszahl, eine **Schicht**, nach demselben Muster. Das kann `numpy` für
die ganze Schicht auf einmal tun, statt Zustand für Zustand:

- Ein Zustand `(a, b)` wird als eine Zahl `a * N + b` gespeichert (`N` ist die Zahl der Felder eines Labyrinths).
- Die Zugtabellen sind numpy-Arrays. `tab1[r][a]` mit einem Array `a` liefert die neuen Felder für alle Zustände der Schicht.
- `gesehen` ist ein Array mit einem Wahrheitswert (1 Byte) pro möglichem Zustand.
- Die nächste Schicht besteht aus den neuen Zuständen für alle vier Anweisungen, ohne Doppelte (`np.unique`) und ohne bereits gesehene.
- Den Weg findet man **rückwärts**: Man merkt sich alle Schichten und sucht für den Zielzustand in der vorletzten Schicht einen
  Zustand, der mit einer Anweisung dorthin führt, und so weiter.

Das braucht nur ein Array von `N²` Wahrheitswerten und je Schicht ein Array. Bei 101 × 101 Feldern sind das rund 100 MB und ein paar
hundert MB für die Schichten.

In [ ]:
import numpy as np


def breitensuche_numpy(tab1, tab2, schichten_merken=True):
    N = len(tab1[0])
    t1 = np.array([[0 if f is None else f for f in zeile] for zeile in tab1], dtype=np.int64)
    t2 = np.array([[0 if f is None else f for f in zeile] for zeile in tab2], dtype=np.int64)
    gesehen = np.zeros(N * N, dtype=bool)
    gesehen[0] = True
    schicht = np.array([0], dtype=np.int64)
    schichten = [schicht]
    tiefe = 0
    anzahl = 1
    while not gesehen[N * N - 1]:
        a = schicht // N
        b = schicht % N
        kandidaten = np.concatenate([t1[r][a] * N + t2[r][b] for r in range(4)])
        kandidaten = np.unique(kandidaten)
        schicht = kandidaten[~gesehen[kandidaten]]
        if len(schicht) == 0:
            return None, anzahl, schichten                   # Ziel nicht erreichbar
        gesehen[schicht] = True
        anzahl += len(schicht)
        tiefe += 1
        if schichten_merken:
            schichten.append(schicht.astype(np.int32))
    return tiefe, anzahl, schichten


def weg_numpy(schichten, tab1, tab2):
    N = len(tab1[0])
    t1 = np.array([[0 if f is None else f for f in zeile] for zeile in tab1], dtype=np.int64)
    t2 = np.array([[0 if f is None else f for f in zeile] for zeile in tab2], dtype=np.int64)
    zustand = N * N - 1
    folge = []
    for k in range(len(schichten) - 1, 0, -1):
        davor = schichten[k - 1].astype(np.int64)
        a = davor // N
        b = davor % N
        for r in range(4):
            treffer = np.flatnonzero(t1[r][a] * N + t2[r][b] == zustand)
            if len(treffer) > 0:
                zustand = int(davor[treffer[0]])
                folge.append(r)
                break
    folge.reverse()
    return folge


for nr in [0, 1, 2, 3]:
    n, m, tab1, tab2 = lade('labyrinthe%d' % nr)
    tiefe, anzahl, schichten = breitensuche_numpy(tab1, tab2)
    folge = weg_numpy(schichten, tab1, tab2)
    print('labyrinthe%d: %d Anweisungen, %d Zustände gesehen, nachgespielt: %s' % (
        nr, tiefe, anzahl, fuehre_aus(folge, tab1, tab2) == (n * m - 1, n * m - 1)))

labyrinthe0: 8 Anweisungen, 31 Zustände gesehen, nachgespielt: True
labyrinthe1: 31 Anweisungen, 342 Zustände gesehen, nachgespielt: True
labyrinthe2: 65 Anweisungen, 8195 Zustände gesehen, nachgespielt: True
labyrinthe3: 164 Anweisungen, 235671 Zustände gesehen, nachgespielt: True


Die Längen sind dieselben wie bei der Python-Breitensuche. (Die Zahl der gesehenen Zustände ist um bis zu 35 kleiner, weil `breitensuche_numpy`
sofort stoppt, wenn das Ziel gesehen wird, die Breitensuche oben aber erst, wenn sie das Ziel aus der Warteschlange nimmt.) Mit dieser Methode kann man auch die großen Beispiele rechnen. Das sind die
Ergebnisse (gemessen mit dem Programm, das diese Seite erzeugt hat; die Ausführung dauert bei Beispiel 4 etwa eine halbe Minute, bei
den anderen mehrere Minuten, und braucht je nach Beispiel 0,5 bis 1,5 GB Speicher, deshalb steht sie nicht als Zelle hier):

| Beispiel | Größe | Anweisungen | Hinweise | Zustände besucht | davon möglich |
|---|---|---|---|---|---|
| 4 | 101 × 101 | 14384 | 14384 | 91 540 337 | 104 060 401 |
| 5 | 151 × 151 | 1308 | 1308 | 347 626 044 | 519 885 601 |
| 8 | 150 × 150 | 472 | 472 | 371 701 448 | 506 250 000 |
| 9 | 170 × 170 | 1012 | 1012 | 760 411 619 | 835 210 000 |

Alle vier Längen stimmen mit den Lösungshinweisen überein. Die Folge von Beispiel 4 haben wir (mit `weg_numpy`) auch nachgespielt: Sie
führt beide Personen ans Ziel. So würdest du Beispiel 4 selbst ausprobieren (auf dem eigenen Rechner, nicht im Browser):

```python
n, m, tab1, tab2 = lade('labyrinthe4')
tiefe, anzahl, schichten = breitensuche_numpy(tab1, tab2)
folge = weg_numpy(schichten, tab1, tab2)
print(tiefe, anzahl, fuehre_aus(folge, tab1, tab2) == (n * m - 1, n * m - 1))
```

Für die Beispiele 5, 8 und 9 lässt man `schichten_merken=False` weg und bekommt nur die Länge (dann braucht man nur das Array
`gesehen` und eine Schicht im Speicher). Für den Weg selbst bräuchte man ein cleveres Verfahren, das die Schichten nicht alle speichert
(zum Beispiel zweimal rechnen und nur jede 100. Schicht merken).

**Warum so viele Zustände?** Bei diesen Beispielen sind die Labyrinthe so lang gewunden, dass fast jedes Paar von Feldern
irgendwann als Zustand vorkommt. Dann nützt auch A\* kaum etwas, denn auch A\* muss fast alle Zustände aufmachen (die Hinweise
zeigen: A\* 5 s, Breitensuche 8 s bei Beispiel 4). Bei Beispiel 6 ist es anders: Dort muss A\* nur einen winzigen Teil der 3,9 Milliarden
möglichen Zustände aufmachen.

## Zum Schluss: dasselbe mit MiniZinc (und warum es hier an Grenzen stößt)

Man kann die Anweisungsfolge auch als Constraint-Problem beschreiben. Dazu legt man eine **feste Höchstzahl `T`** von Anweisungen
fest und fragt nach einer Folge `d[1..T]`, bei der beide Personen am Ende im Ziel stehen. Die Position nach jeder Anweisung ist ein
Tabellenzugriff: `p[t] = z1[d[t], p[t-1]]`. Weil im Ziel jede Anweisung wirkungslos ist, zählt man nur die Anweisungen, vor denen noch
jemand außerhalb des Ziels steht, und **minimiert** diese Zahl. Die Zugtabellen `z1` und `z2` kommen aus Python.

```minizinc
int: N;                                  % Felder je Labyrinth (Start = 1, Ziel = N)
int: T;                                  % obere Grenze für die Zahl der Anweisungen
array[1..4, 1..N] of int: z1;            % Zugtabellen (Felder ab 1 nummeriert), 1 = ↑, 2 = →, 3 = ↓, 4 = ←
array[1..4, 1..N] of int: z2;
array[0..T] of var 1..N: p;              % Feld der ersten Person nach t Anweisungen
array[0..T] of var 1..N: q;              % Feld der zweiten Person
array[1..T] of var 1..4: d;              % die Anweisungen
constraint p[0] = 1 /\ q[0] = 1;
constraint forall(t in 1..T)(p[t] = z1[d[t], p[t-1]] /\ q[t] = z2[d[t], q[t-1]]);
constraint p[T] = N /\ q[T] = N;
var 0..T: laenge = sum(t in 0..T-1)(bool2int(p[t] != N \/ q[t] != N));
solve minimize laenge;
output ["laenge = ", show(laenge), "\n"];
```

Die Datendatei erzeugt eine Python-Zelle aus den Zugtabellen (in MiniZinc beginnen Felder bei 1, bei Gruben tragen wir irgendeinen
Wert ein, denn dort steht nie jemand).

In [ ]:
def als_dzn(nr, T):
    n, m, tab1, tab2 = lade('labyrinthe%d' % nr)
    felder = n * m
    teile = ['N = %d;' % felder, 'T = %d;' % T]
    for k, tab in enumerate([tab1, tab2]):
        werte = []
        for r in range(4):
            for feld in range(felder):
                neu = tab[r][feld]
                werte.append('1' if neu is None else str(neu + 1))
        teile.append('z%d = array2d(1..4, 1..%d, [%s]);' % (k + 1, felder, ', '.join(werte)))
    return '\n'.join(teile)


print(als_dzn(0, 10))

N = 9;
T = 10;
z1 = array2d(1..4, 1..9, [1, 2, 3, 1, 2, 3, 4, 5, 9, 1, 3, 3, 4, 5, 6, 8, 8, 9, 4, 5, 6, 7, 8, 9, 7, 8, 9, 1, 2, 2, 4, 5, 6, 7, 7, 9]);
z2 = array2d(1..4, 1..9, [1, 2, 3, 1, 2, 3, 1, 8, 9, 1, 3, 3, 5, 5, 6, 1, 9, 9, 4, 5, 6, 4, 5, 9, 1, 8, 9, 1, 2, 2, 4, 4, 6, 1, 1, 9]);


Mit dem Solver **Gecode** (in der MiniZinc IDE auswählen) ergibt sich für `labyrinthe0` mit `T = 10` und für `labyrinthe1` mit `T = 35`:

**Ausgabe:**

```
laenge = 8
----------
==========
```

**Ausgabe:**

```
laenge = 31
----------
==========
```

Das sind dieselben Längen 8 und 31 wie oben, und der Solver beweist, dass es nicht kürzer geht. **Das war es aber auch schon:** Wir haben
das Modell mit den Solvern Gecode, Chuffed und CP-SAT auch an den anderen Beispielen probiert. Bei `labyrinthe2` (65 Anweisungen,
`T = 70`) und bei `labyrinthe3` (164 Anweisungen, `T = 170`) lieferte keiner von ihnen innerhalb von 60 Sekunden ein bewiesenes Ergebnis.
Die Beispiele 4 bis 9 haben mehr als 1000 Anweisungen bzw. Hunderte Millionen Zustände. Dort ist das Modell erst recht hoffnungslos, und bei
`labyrinthe7` kennt man `T` gar nicht. Der Grund: Die Breitensuche *merkt sich* jeden Zustand und besucht ihn nur einmal. Der
Solver sucht dagegen Folgen aus Anweisungen und kann gleiche Zustände auf verschiedenen Wegen schlecht erkennen. Für diese Aufgabe
ist die Suche im Zustandsraum das passende Werkzeug und MiniZinc nur bei sehr kleinen Fällen brauchbar.

## Zusammenfassung

- Ein **Zustand** ist das Paar der beiden Felder. Wände, Gruben und Ziel stecken in einer **Zugtabelle** je Labyrinth. Die Gruben
  (Aufgabe b) kosten dann nichts Zusätzliches.
- **Breitensuche** über die Zustände findet eine kürzeste Anweisungsfolge. Sie stimmt bei 0, 1, 2, 3 und 7 mit den Hinweisen überein.
- **A\*** mit dem größeren der beiden Einzelabstände als Schranke braucht weniger Zustände und löst Beispiel 6.
- Die Beispiele 4, 5, 8 und 9 haben so viele Zustände, dass reines Python scheitert. Schichtweise mit `numpy` geht es, Längen wie in
  den Hinweisen.
- **MiniZinc** (feste Höchstlänge `T`) schafft nur die kleinsten Beispiele.
- Bei `labyrinthe0` nennen die Hinweise 2 optimale Lösungen, wir finden beim vollständigen Durchprobieren genau 1. Verlangt war die Zahl nicht.